In [3]:
import pandas as pd
import numpy as np

# Define a list of standard chromosomes
standard_chromosomes = ['chr' + str(i) for i in range(1, 23)] + ['chrX', 'chrY']

# Read the VCF file into a DataFrame, extracting the necessary columns
vcf_file_path = '../carcinoma_normal_sniffles_filter/sniffles/Panc1.vcf'
vcf_df = pd.read_csv(vcf_file_path, sep='\t', comment='#', header=None)
vcf_df.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE']

# Define function to parse the INFO field
def parse_info(info_str):
    info_dict = {}
    for item in info_str.split(';'):
        if '=' in item:
            key, value = item.split('=')
            info_dict[key] = value
        else:
            info_dict[item] = True
    return info_dict

# Initialize lists to store the extracted information for BNDs
chrom1_list = []
start1_list = []
chrom2_list = []
start2_list = []

# Extract relevant information from the VCF data
for index, row in vcf_df.iterrows():
    info = parse_info(row['INFO'])
    svtype_value = info.get('SVTYPE')
    chrom1 = row['CHROM']

    # Check if the chromosome is standard and SVTYPE is BND
    if svtype_value == 'BND' and chrom1 in standard_chromosomes:
        chrom1_list.append(chrom1)
        start1_list.append(row['POS'])

        alt = row['ALT']
        # Parse the ALT field to get the second chromosomal location
        if ']' in alt or '[' in alt:
            parts = alt.replace(']', '[').split('[')
            if len(parts) >= 2:
                chrom2_loc = parts[1]
                chrom2_parts = chrom2_loc.split(':')
                if len(chrom2_parts) == 2:
                    chrom2 = chrom2_parts[0]
                    start2 = chrom2_parts[1]
                    chrom2_list.append(chrom2)
                    start2_list.append(start2)
                else:
                    chrom2_list.append(np.nan) # Replace with NaNs if there is no information
                    start2_list.append(np.nan)
            else:
                chrom2_list.append(np.nan)
                start2_list.append(np.nan)
        else:
            chrom2_list.append(np.nan)
            start2_list.append(np.nan)

# Create a DataFrame with the extracted information
bnd_df = pd.DataFrame({
    'CHROM1': chrom1_list,
    'START1': start1_list,
    'CHROM2': chrom2_list,
    'START2': start2_list
})

# Filter to only include rows where CHROM2 is also a standard chromosome
bnd_df = bnd_df[bnd_df['CHROM2'].isin(standard_chromosomes)]

# Save the DataFrame to a CSV file
output_csv_path = '../carcinoma_normal_sniffles_filter/complex_csv/Panc1.csv'
bnd_df.to_csv(output_csv_path, index=False)

# Print the number of BNDs
print(f'Number of BND SV types in standard chromosomes: {bnd_df.shape[0]}')
print(f'CSV file with BNDs of standard chromosomes saved to: {output_csv_path}')

Number of BND SV types in standard chromosomes: 262
CSV file with BNDs of standard chromosomes saved to: ../carcinoma_normal_sniffles_filter/complex_csv/Panc1.csv


In [11]:
import pandas as pd

df = pd.read_csv("csv_files/long-range-variants/PD51_CTS_P25_LR.csv")
# Filter the DataFrame to include only specific chromosomes
specific_chromosomes = ['chr9', 'chr17', 'chr18']
filtered_df = df[df['CHROM1'].isin(specific_chromosomes)].copy()

# Display the filtered DataFrame
filtered_df_csv = 'csv_files/long-range-variants/filtered_chr/PD51_CTS_P25_LR_filtered.csv'
filtered_df.to_csv(filtered_df_csv, index=False)